<a href="https://colab.research.google.com/github/jihun0251/korean-spam-classifierML/blob/main/%EC%8A%A4%ED%8C%B8_%EB%B6%84%EB%A5%98_%EB%A1%9C%EC%A7%80%EC%8A%A4%ED%8B%B1_%ED%9A%8C%EA%B7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 라이브러리 불러오기

In [ ]:
!pip install konlpy
import pandas as pd  # 데이터를 표처럼 다루는 도구
from sklearn.feature_extraction.text import TfidfVectorizer  # 텍스트를 TF - IDF 숫자 백터로 바꿔주는 도구
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델
from sklearn.model_selection import train_test_split  # 데이터를 학습용/시험용으로 나워주는 모델
from sklearn.metrics import classification_report, confusion_matrix  # 성능을 평가하는 도구들
from konlpy.tag import Okt  # 한국어 형태소 분석기

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 34.9 MB/s eta 0:00:00


In [ ]:
okt = Okt()  # Okt 형태소 분석기 준비
print(okt.morphs("무료 쿠폰을 지금 받으세요"))  # 형태소 단위로 쪼개보기

['무료', '쿠폰', '을', '지금', '받으세요']


# 데이터 불러오기
- 이제 SMS Spam 데이터를 가져온다. 다행히 인터넷에 공개돼 있어서 URL로 바로 읽어올 수 있다.
- pandas의 read_scv 가 이 일을 한다.

In [ ]:
# 스팸 문자 50개 - 다양한 유형
spam_messages = [
    "축하합니다! 100만원에 당첨되셨습니다 지금 클릭 http://bit.ly/win123",
    "[Web발신] 국세청 환급금 조회 안내 http://tax-refund.xyz",
    "택배 배송 주소가 불분명합니다 정보 입력 http://delivery-check.com",
    "엄마 나 폰 고장나서 그런데 이 링크로 인증 좀 해줘 http://help-me.net",
    "고객님 카드가 해외에서 결제되었습니다 본인 아니면 클릭 http://card-stop.co",
    "무료 상품권 증정 이벤트! 지금 신청 http://free-gift.shop",
    "대출 한도 5000만원 승인 완료 지금 확인 http://easy-loan.kr",
    "[긴급] 계정이 정지되었습니다 즉시 인증 http://account-fix.io",
    "당신만을 위한 특별 할인 쿠폰 도착 http://coupon-now.link",
    "건강검진 결과 이상 소견 확인하려면 클릭 http://health-check.vip",
    "[국민은행] 본인인증 필요 아래 링크 접속 http://kb-auth.xyz",
    "경품 추첨에 당첨되셨어요 배송지 입력 http://prize-win.net",
    "스미싱 신고 안내 클릭하여 확인하세요 http://report-fake.co",
    "월 300만원 재택 부업 지금 시작 http://home-job.shop",
    "고객님의 택배가 보관중입니다 http://parcel-hold.xyz",
    "비트코인 투자로 하루 50만원 수익 http://coin-rich.io",
    "[카카오] 비정상 로그인 감지 확인 http://kakao-check.link",
    "신용등급 무료 조회 이벤트 http://credit-free.kr",
    "당첨을 축하드립니다 신세계상품권 50만원 http://sse-gift.net",
    "긴급 연락 바랍니다 아버지 사고 병원비 송금 http://hospital-pay.co",
    "넷플릭스 결제 실패 정보 갱신 필요 http://netflix-fix.xyz",
    "회원님 포인트 소멸 예정 지금 사용 http://point-use.shop",
    "정부지원금 신청 대상자 확인 http://gov-support.io",
    "명품 시계 90% 할인 한정 판매 http://luxury-sale.link",
    "로또 1등 번호 무료 공개 http://lotto-win.kr",
    "고객님 명의로 휴대폰이 개통되었습니다 http://phone-open.net",
    "카드 포인트 현금 전환 마지막 기회 http://card-cash.co",
    "[쿠팡] 미수령 택배 주소 확인 http://coupang-fake.xyz",
    "초대장이 도착했습니다 확인 http://invite-card.shop",
    "귀하의 계좌가 보이스피싱에 연루 http://account-safe.io",
    "성인인증 무료 가입 이벤트 http://adult-free.link",
    "재난지원금 25만원 신청하세요 http://disaster-fund.kr",
    "당신의 사진이 유출되었습니다 확인 http://photo-leak.net",
    "[농협] 보안카드 재발급 안내 http://nh-security.co",
    "급전 필요하신가요 무담보 당일대출 http://quick-money.xyz",
    "프리미엄 회원 무료 체험 신청 http://premium-free.shop",
    "택배기사입니다 부재중 연락주세요 http://delivery-call.io",
    "주식 리딩방 무료 입장 수익률 300% http://stock-room.link",
    "고객센터입니다 환불 처리 정보 입력 http://refund-now.kr",
    "당첨 안내 갤럭시 최신폰 무료 증정 http://galaxy-free.net",
    "법원 출석 통지서 확인 바랍니다 http://court-notice.co",
    "카지노 첫충전 100% 보너스 http://casino-bonus.xyz",
    "건강보험 환급금 조회하세요 http://insurance-back.shop",
    "택배 통관 보류 관세 결제 필요 http://customs-pay.io",
    "유명 쇼핑몰 폐업 정리 90% 세일 http://closing-sale.link",
    "본인인증만 하면 5만원 즉시 지급 http://verify-cash.kr",
    "코인 에어드랍 무료 토큰 받기 http://airdrop-free.net",
    "미납 요금이 있습니다 즉시 납부 http://unpaid-bill.co",
    "귀하는 배심원으로 선정되었습니다 http://jury-select.xyz",
    "한정수량 아이폰 추첨 응모 http://iphone-draw.shop",
]

# 정상 문자 50개 - 일상 대화, 공식 안내
ham_messages = [
    "엄마 나 오늘 저녁 약속 있어서 늦어",
    "내일 회의 10시로 변경됐어요 참고 부탁드려요",
    "고객님 통신요금 할인 이벤트 당첨 안내 고객센터로 문의주세요",
    "택배가 오늘 오후에 도착 예정입니다 부재시 경비실에 맡겨드릴게요",
    "주문하신 상품이 배송 시작되었습니다 마이페이지에서 확인하세요",
    "지훈아 이번 주말에 같이 밥 먹자",
    "결제가 정상적으로 완료되었습니다 이용해주셔서 감사합니다",
    "도서관 대출 도서 반납일이 내일까지입니다",
    "축하해 합격 소식 들었어 정말 잘됐다",
    "이번 달 카드 명세서가 발행되었습니다 앱에서 확인 가능합니다",
    "오늘 점심 뭐 먹을까 메뉴 추천 좀",
    "회의 자료 메일로 보냈으니 확인 부탁해요",
    "엄마 김치 떨어졌는데 좀 보내줄 수 있어?",
    "내일 비 온대 우산 챙겨가",
    "프로젝트 마감 다음주 금요일까지인거 잊지마",
    "생일 축하해 선물은 내가 준비했어",
    "병원 예약 내일 오후 3시인거 기억하지?",
    "엄마 나 시험 잘 봤어 걱정 마",
    "오늘 야근이라 저녁 먼저 먹어",
    "주말에 영화 볼래? 예매할게",
    "택배 잘 받았어 고마워",
    "회사 워크샵 다음달 둘째주로 정해졌어요",
    "할머니 댁에 언제 갈지 정하자",
    "운동 같이 하기로 한거 오늘 맞지?",
    "리포트 다 썼어 한번 봐줄래?",
    "고객님 예약이 정상 접수되었습니다",
    "다음 정기검진은 6개월 후입니다",
    "강의 자료 업로드 했으니 확인하세요",
    "이번 학기 수강신청 잘 됐어?",
    "퇴근하고 마트 들러서 갈게 뭐 필요해?",
    "동아리 모임 이번주 수요일 저녁이야",
    "면접 잘 보고 와 응원할게",
    "주문번호 조회되었습니다 내일 출고 예정입니다",
    "친구야 오랜만이다 언제 한번 보자",
    "엄마 용돈 보내주셔서 감사해요",
    "발표 준비 거의 다 끝났어",
    "내일 아침 일찍 출발하자 7시까지 와",
    "감기 조심하고 푹 쉬어",
    "계약서 검토 후 회신 드리겠습니다",
    "오늘 경기 봤어? 진짜 역전승이었어",
    "도서 연체료가 부과되었으니 반납 바랍니다",
    "새 학기 시간표 나왔어 같이 확인하자",
    "택배 수령 확인 부탁드립니다",
    "주말 잘 보내고 월요일에 보자",
    "엄마 나 집 가는 길이야 곧 도착해",
    "팀 회식 장소 예약 완료했습니다",
    "건강하게 잘 지내고 있지? 보고싶다",
    "과제 제출 확인 메일 받았어요 감사합니다",
    "내일 등산 가는데 같이 갈래?",
    "수고했어 오늘 정말 고생 많았어",
]

import pandas as pd
df = pd.DataFrame({
    "message": spam_messages + ham_messages,
    "label": ["spam"] * len(spam_messages) + ["ham"] * len(ham_messages)
})

print("전체 개수:", len(df))
print(df["label"].value_counts())
df.head()

전체 개수: 100
label
spam    50
ham     50
Name: count, dtype: int64


,message,label
0,축하합니다! 100만원에 당첨되셨습니다 지금 클릭 http://bit.ly/win123,spam
1,[Web발신] 국세청 환급금 조회 안내 http://tax-refund.xyz,spam
2,택배 배송 주소가 불분명합니다 정보 입력 http://delivery-check.com,spam
3,엄마 나 폰 고장나서 그런데 이 링크로 인증 좀 해줘 http://help-me.net,spam
4,고객님 카드가 해외에서 결제되었습니다 본인 아니면 클릭 http://card-sto...,spam


# 3. X,y 나누고 학습/시험으로 분할

In [ ]:
# 입력(X)과 정답(y) 나누기
X = df["message"]   # 입력: 문자 내용
y = df["label"]     # 정답: spam / ham

# 학습용 / 시험용으로 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 전체의 20%를 시험용으로
    stratify=y,         # spam:ham 비율을 양쪽에 똑같이 유지
    random_state=42     # 매번 같은 결과가 나오도록 고정
)

# 잘 나뉘었는지 확인
print("학습용 개수:", len(X_train))
print("시험용 개수:", len(X_test))
print("\n학습용 라벨 분포:")
print(y_train.value_counts())
print("\n시험용 라벨 분포:")
print(y_test.value_counts())

학습용 개수: 80
시험용 개수: 20

학습용 라벨 분포:
label
ham     40
spam    40
Name: count, dtype: int64

시험용 라벨 분포:
label
spam    10
ham     10
Name: count, dtype: int64


# 4. 한국어용 토크나이저 함수 만들기

In [ ]:
# Okt 형태소 분석기 준비 (앞에서 만들었지만 안전하게 다시)
okt = Okt()

# 문장 하나를 받아서 형태소 리스트로 돌려주는 함수
def okt_tokenizer(text):
    return okt.morphs(text)   # morphs = 형태소(morpheme) 단위로 쪼개기

# 테스트: 우리 데이터의 첫 문장으로 확인해보기
print(okt_tokenizer("축하합니다! 100만원에 당첨되셨습니다 지금 클릭"))

['축하', '합니다', '!', '100만원', '에', '당첨', '되셨습니다', '지금', '클릭']


In [ ]:
# TfidfVectorizer에 우리가 만든 한국어 토크나이저(okt_tokenizer)를 연결
vectorizer = TfidfVectorizer(tokenizer=okt_tokenizer)

# 학습용 데이터로 "단어 사전"을 만들고(fit) + 숫자로 변환(transform)
# fit_transform = 이 둘을 한 번에
X_train_vec = vectorizer.fit_transform(X_train)

# 시험용 데이터는 변환만(transform). 사전은 학습용 걸 그대로 씀
X_test_vec = vectorizer.transform(X_test)  # 학습용에 fit이 없는 이유: fit은 단어사전을 학습을 통해 만드는 것인데, 시험용에 fit이 들어가면 제대로된 평가를 못하게 됨

# 변환 결과 모양 확인
print("학습용 행렬 모양:", X_train_vec.shape)
print("시험용 행렬 모양:", X_test_vec.shape)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


학습용 행렬 모양: (80, 403)
시험용 행렬 모양: (20, 403)


# 5. 모델 학습
- 숫자 행렬이 된 데이터를 로지스틱 회귀 모델에게 학습시킬 차례이다.

In [ ]:
# 로지스틱 회귀 모델 준비
model = LogisticRegression()

# 학습! (숫자로 바뀐 학습용 데이터 + 정답 라벨을 보여줌)
model.fit(X_train_vec, y_train)

print("학습 완료!")

학습 완료!


# 6. 모델 평가

In [ ]:
# 시험용 데이터(숫자로 변환해둔 것)로 예측
y_pred = model.predict(X_test_vec)

# 예측 결과와 실제 정답을 나란히 비교
print("실제 정답:", list(y_test))
print("모델 예측:", list(y_pred))

# 상세 성능 리포트 (정밀도, 재현율, F1)
print("\n성능 리포트:")
print(classification_report(y_test, y_pred))

# 혼동행렬 (무엇을 무엇으로 착각했는지)
print("혼동행렬:")
print(confusion_matrix(y_test, y_pred))

실제 정답: ['spam', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'ham', 'ham', 'ham', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam', 'ham', 'spam', 'spam']
모델 예측: ['spam', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'ham', 'ham', 'spam', 'spam']

성능 리포트:
              precision    recall  f1-score   support

         ham       0.82      0.90      0.86        10
        spam       0.89      0.80      0.84        10

    accuracy                           0.85        20
   macro avg       0.85      0.85      0.85        20
weighted avg       0.85      0.85      0.85        20

혼동행렬:
[[9 1]
 [2 8]]


# 7. 새 문자 넣어보기

In [ ]:
# 새로운 문자 메시지들 (모델이 한 번도 본 적 없는 것)
new_messages = [
    "축하합니다 고객님 경품 당첨 지금 링크 클릭 http://prize.win",
    "오늘 저녁에 시간 되면 같이 영화 볼래?",
    "[국민은행] 본인인증 필요 아래 링크 접속 http://kb-auth.xyz",
    "내일 과제 제출 마감인거 잊지마",
]

# 새 문자도 같은 방식으로 숫자 변환 (transform만!)
new_vec = vectorizer.transform(new_messages)

# 예측
predictions = model.predict(new_vec)

# 결과를 보기 좋게 출력
for msg, pred in zip(new_messages, predictions):
    print(f"[{pred}] {msg}")

[spam] 축하합니다 고객님 경품 당첨 지금 링크 클릭 http://prize.win
[ham] 오늘 저녁에 시간 되면 같이 영화 볼래?
[spam] [국민은행] 본인인증 필요 아래 링크 접속 http://kb-auth.xyz
[ham] 내일 과제 제출 마감인거 잊지마


In [ ]:
# 각 클래스에 대한 확률 구하기
probabilities = model.predict_proba(new_vec)

# 클래스 순서 확인 (보통 ['ham', 'spam'] 알파벳 순)
print("클래스 순서:", model.classes_)
print()

# 문자별로 spam일 확률 보여주기
for msg, prob in zip(new_messages, probabilities):
    # model.classes_에서 'spam'이 몇 번째인지 찾기
    spam_idx = list(model.classes_).index("spam")
    spam_prob = prob[spam_idx]
    print(f"스팸 확률 {spam_prob*100:.1f}% | {msg}")

클래스 순서: ['ham' 'spam']

스팸 확률 64.5% | 축하합니다 고객님 경품 당첨 지금 링크 클릭 http://prize.win
스팸 확률 31.0% | 오늘 저녁에 시간 되면 같이 영화 볼래?
스팸 확률 67.4% | [국민은행] 본인인증 필요 아래 링크 접속 http://kb-auth.xyz
스팸 확률 38.2% | 내일 과제 제출 마감인거 잊지마
